*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 13: Advanced Scale: Mixed Precision & Multi-GPU. It covers numeric precision choices, DDP coordination, and the batch-size decisions that matter when moving beyond a single device.

Scaling a training job is not only about more devices. It also requires a careful understanding of precision, throughput, and the effective global batch size seen by the optimizer.

## Automatic Mixed Precision
### Step 0: Importing Previously Implemented Classes

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pytorch_lightning as pl

# Precision settings change numeric behavior and throughput, so they should be chosen intentionally and validated early.
class VisionDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir: str = "./data",
        batch_size: int = 256,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = min(4, os.cpu_count() or 1)
        self.pin_memory = torch.cuda.is_available()

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5),
            ),
        ])

    def prepare_data(self):
        datasets.CIFAR10(
            self.data_dir, train=True, download=True
        )
        datasets.CIFAR10(
            self.data_dir, train=False, download=True
        )

    def setup(self, stage: str | None = None):
        if stage in ("fit", "validate", None):
            if not hasattr(self, "cifar_train"):
                full_dataset = datasets.CIFAR10(
                    self.data_dir,
                    train=True,
                    transform=self.transform,
                )
                generator = torch.Generator().manual_seed(42)
                self.cifar_train, self.cifar_val = random_split(
                    full_dataset,
                    [45000, 5000],
                    generator=generator,
                )

        if stage in ("test", "predict", None):
            if not hasattr(self, "cifar_test"):
                self.cifar_test = datasets.CIFAR10(
                    self.data_dir,
                    train=False,
                    transform=self.transform,
                )

    def _make_loader(self, dataset, shuffle):
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.num_workers > 0,
        )

    def train_dataloader(self):
        return self._make_loader(self.cifar_train, shuffle=True)

    def val_dataloader(self):
        return self._make_loader(self.cifar_val, shuffle=False)

    def test_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)

    def predict_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)
    

class LitCIFARClassifier(pl.LightningModule):
    def __init__(self, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.backbone(x)

    def _shared_step(self, batch, prefix):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log(
            f"{prefix}_loss",
            loss,
            on_epoch=True,
            prog_bar=True,
            sync_dist=prefix != "train",
            batch_size=x.size(0),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return self(x).argmax(dim=1)

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr
        )

### Step 2: Select a Precision Policy from Hardware Support

In [ ]:
# Precision selection is hardware-dependent. FP32 is the safe baseline everywhere.
# BF16 retains FP32's exponent range but lower fractional precision—ideal for large models.
# FP16 has higher fractional precision but a narrower range and requires loss scaling.
# Choose the best available precision for the hardware.
import torch

use_cuda = torch.cuda.is_available()

# BF16 (bfloat16) is preferred on supported CUDA hardware because it:
# - Matches FP32's exponent width (8 bits), avoiding underflow/overflow
# - Does not require dynamic loss scaling
# - Works well for large models and distributed training
if use_cuda and torch.cuda.is_bf16_supported():
    precision = "bf16-mixed"
# FP16 (float16) is a fallback on older CUDA GPUs; it has a narrower range.
elif use_cuda:
    precision = "16-mixed"
# CPU and MPS backends default to full FP32.
else:
    precision = "32-true"

assert precision in {"bf16-mixed", "16-mixed", "32-true"}


### Step 3: Validate the Policy Before Scaling Out

In [5]:
# Before scaling to multiple devices, validate the chosen precision on a single device.
# This separates precision failures from distributed coordination issues.
# A fast_dev_run with one batch quickly exposes numerical instability or kernel incompatibility.
import pytorch_lightning as pl

precision_model = LitCIFARClassifier(lr=1e-3)
precision_data = VisionDataModule(
    data_dir="./data",
    batch_size=64,
)

# fast_dev_run=True runs one batch and verifies that forward, backward, and optimizer steps work
# with the selected precision before committing to a longer training run.
precision_trainer = pl.Trainer(
    fast_dev_run=True,
    accelerator="auto",
    devices=1,
    precision=precision,  # Test the selected precision policy
    logger=False,
    enable_checkpointing=False,
)

precision_trainer.fit(
    precision_model,
    datamodule=precision_data,
)

# Success here means the precision policy is compatible with the hardware and model.
assert precision_trainer.global_step == 1


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


## Distributed Data Parallel Architecture

### Step 2: Calculate Local, Global & Effective Batch Size

In [ ]:
# The effective global batch size is the number of samples that contribute to one optimizer step.
# This is critical because it affects gradient noise, convergence speed, and memory usage.
# Formula: local_batch × (devices_per_node × num_nodes) × accumulation_steps
# Changing the world size without adjusting learning rate and schedule can destabilize training.

def effective_batch_size(
    local_batch_size,
    devices_per_node,
    num_nodes,
    accumulation_steps,
):
    # world_size is the total number of devices across all nodes
    world_size = devices_per_node * num_nodes
    # Total effective batch multiplies local batch, world size, and accumulation
    return (
        local_batch_size
        * world_size
        * accumulation_steps
    )

# Example: 64 local batch × 8 devices/node × 2 nodes × 2 accum = 2048 samples per step
planned_batch = effective_batch_size(
    local_batch_size=64,
    devices_per_node=8,
    num_nodes=2,
    accumulation_steps=2,
)

# Always verify the effective batch size before launching expensive distributed runs.
assert planned_batch == 2048


## Configuring Distributed Training in Lightning

### Step 1: Scale within the Available Machine

In [7]:
# Start with local multi-GPU before requesting a cluster. This validates distributed code
# without infrastructure overhead and exposes process launch and communication issues early.
available_gpus = torch.cuda.device_count()
# Use up to 2 local GPUs for testing (smaller scale than full cluster).
local_devices = min(2, available_gpus)

if local_devices >= 2:
    # If multiple GPUs are available, use DDP (DistributedDataParallel) strategy.
    accelerator = "gpu"
    strategy = "ddp"  # One process per GPU, synchronized gradients
    devices = local_devices
else:
    # Fall back to auto-selection if only one GPU or no GPU.
    accelerator = "auto"
    strategy = "auto"
    devices = 1

# fast_dev_run=True runs one batch to validate distributed setup without full training.
local_scale_trainer = pl.Trainer(
    accelerator=accelerator,
    devices=devices,
    strategy=strategy,
    precision=precision,
    fast_dev_run=True,        # Quick validation of distributed coordination
    logger=False,
    enable_checkpointing=False,
)

local_scale_model = LitCIFARClassifier(lr=1e-3)
local_scale_data = VisionDataModule(
    data_dir="./data",
    batch_size=64,
)

local_scale_trainer.fit(
    local_scale_model,
    datamodule=local_scale_data,
)

# Success here means DDP process launch, gradient sync, and metrics reduction work correctly.
assert local_scale_trainer.global_step == 1


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


### Step 2: Express the Multi-Node Cluster Plan

In [ ]:
# Multi-node distributed training: each of 2 nodes runs 8 GPUs, for a world size of 16.
# Effective global batch: 64 × 16 × 2 = 2048 samples per optimizer step.
# The surrounding infrastructure (SLURM, Kubernetes, etc.) must handle process launch and networking.
# If you do not have access to multiple nodes, you can skip this section and focus on the single-node version above.

local_batch_size = 64
devices_per_node = 8
num_nodes = 2
accumulation_steps = 2

cluster_model = LitCIFARClassifier(lr=1e-3)
cluster_data = VisionDataModule(
    data_dir="./data",
    batch_size=local_batch_size,
)

# Trainer describes the topology but does NOT launch processes. That is the job of the
# surrounding cluster scheduler (SLURM sbatch, Kubernetes, etc.).
cluster_trainer = pl.Trainer(
    accelerator="gpu",                        # Use GPU accelerator
    devices=devices_per_node,                 # 8 GPUs per node
    num_nodes=num_nodes,                      # 2 nodes total
    strategy="ddp",                           # DistributedDataParallel strategy
    precision="bf16-mixed",                   # Use BF16 mixed precision for efficiency
    accumulate_grad_batches=accumulation_steps,  # Accumulate 2 micro-batches before optimization step
    max_epochs=50,
    sync_batchnorm=False,                     # Set to True only if model uses BatchNorm and batches are small
)

cluster_trainer.fit(
    cluster_model,
    datamodule=cluster_data,
)
